# FLAG 2027 - Notebook 05: Which features work best? (trained model, honest CV)

Notebook 04 measured each feature *alone*. What really matters is how well the **face-voice model** works with them. Here we train our best recipe so far (**D_balanced** from Notebook 03) on different feature combinations:

| Combo | Face input | Voice input |
|---|---|---|
| K1_old | VGGFace | ECAPA | (our current system, CodaBench 33.49) |
| K2..K6 | new faces / combinations | ECAPA (+ WavLM) |

Only combos whose feature files exist are run. 5 folds x 3 seeds each, resumable.

**How to run:** T4 GPU, Runtime > Run all. About 20-40 min. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - load the old CSV features + all new .npz features, aligned row by row
import os, json, time, zipfile, copy
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}   # FEAT[feature_name][split] -> array
Xf, y = read_feat(find_one('train_English_faces.csv'), 4096)
Xv, _ = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['vggface'] = {'train': Xf}; FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['vggface'][s] = read_feat(find_one('/' + track + '/', lang + '_test_faces.csv'), 4096)[0]
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['arcface', 'arcface_aligned', 'clip', 'wavlm_sv']:
    p = os.path.join(FEATDIR, name + '.npz')
    if os.path.exists(p):
        d = np.load(p); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}
print('Available features:', {k: v['train'].shape[1] for k, v in FEAT.items()})

y = y.astype(int)
lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}

SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

In [ ]:
# Cell 3 - feature combinations to test (each block is L2-normalised, then blocks are concatenated)
# K1-K7 done on 13 Oct (results are reused from nb05_cv.csv). K8-K10 test whether we can drop the features
# that are hard to re-extract for MAV-Celeb / VoxCeleb (official VGGFace + official ECAPA).
COMBOS = {
    'K1_old':          (['vggface'], ['ecapa']),
    'K2_arcface':      (['arcface_aligned'], ['ecapa']),
    'K3_clip':         (['clip'], ['ecapa']),
    'K4_vgg+arc':      (['vggface', 'arcface_aligned'], ['ecapa']),
    'K5_vgg+clip':     (['vggface', 'clip'], ['ecapa']),
    'K6_allface':      (['vggface', 'arcface_aligned', 'clip'], ['ecapa']),
    'K7_allface+wavlm':(['vggface', 'arcface_aligned', 'clip'], ['ecapa', 'wavlm_sv']),
    'K8_clip|wavlm':   (['clip'], ['wavlm_sv']),                     # only features WE can extract
    'K9_clip+arc|wavlm': (['clip', 'arcface_aligned'], ['wavlm_sv']),  # only features WE can extract
    'K10_clip|ecapa+wavlm': (['clip'], ['ecapa', 'wavlm_sv']),        # drop VGGFace, keep ECAPA
}
COMBOS = {k: v for k, v in COMBOS.items() if all(n in FEAT for n in v[0] + v[1])}
print('Combos that will run:', list(COMBOS))

def build(names, split):
    blocks = []
    for n in names:
        X = FEAT[n][split]
        blocks.append(X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9))
    return np.concatenate(blocks, 1).astype(np.float32)

In [ ]:
# Cell 4 - model (input sizes now flexible), loss, metric
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)

class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)

def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1)
    dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device)
    neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)

def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100

@torch.no_grad()
def same_score(model, xf, xv):
    # HIGHER = same person (official L2 distance between tanh embeddings, negated)
    model.eval()
    mf, sf, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()

In [ ]:
# Cell 5 - honest validation (same folds + pairs as Notebooks 02/03) and the D_balanced training recipe
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk = np.unique(y[idx]); rows = {s: idx[y[idx] == s] for s in spk}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk); V.append(rng.choice(rows[s])); F.append(rng.choice(rows[s])); L.append(1)
        s = rng.choice(spk)
        cand = [t for t in spk if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows[s])); F.append(rng.choice(rows[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)

K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))

SEEDS = [1, 2, 3]
BS, P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE = 128, 16, 8, 2.0, 1e-4, 100, 15

def train_model(XF, XV, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    classes = np.unique(y[tr_idx]); remap = {c: i for i, c in enumerate(classes)}
    ytr_np = np.array([remap[c] for c in y[tr_idx]]); ytr = torch.tensor(ytr_np, device=dev)
    tr = torch.tensor(tr_idx, device=dev)
    norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)   # TRAIN stats only
    mf, sf, mv, sv = norm
    model = FOP(len(classes), XF.shape[1], XV.shape[1]).to(dev); model.norm = norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(classes))]
    steps = len(tr_idx) // BS
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(steps):
            chosen = rng.choice(len(classes), P_ID, replace=False)
            j = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            j = torch.tensor(j, device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep

In [ ]:
# Cell 6 - run all combos (RESUMABLE)
RESDIR = os.path.join(PROJECT, 'results'); os.makedirs(RESDIR, exist_ok=True)
CSV = os.path.join(RESDIR, 'nb05_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold'])
t0 = time.time()
for name, (fn, vn) in COMBOS.items():
    XF = torch.tensor(build(fn, 'train'), device=dev); XV = torch.tensor(build(vn, 'train'), device=dev)
    for seed in SEEDS:
        for k in range(K):
            if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep = train_model(XF, XV, tr_idx, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets]
            row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-17s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))

In [ ]:
# Cell 7 - results table
R = pd.read_csv(CSV)
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('combo').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                    overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab['gap'] = tab['gender'] - tab['std']
tab = tab.loc[[c for c in COMBOS if c in tab.index]].round(2)
note('FEATURE COMBOS (D_balanced recipe, 5-fold CV x 3 seeds; lower is better)')
note(tab.to_string())
BEST = tab['overall'].idxmin()
FINAL_EPOCHS = int(np.median(R[R.combo == BEST]['best_ep']))
note('BEST combo: %s (overall %.2f vs K1_old %.2f) -> final model trains %d epochs'
     % (BEST, tab.loc[BEST, 'overall'], tab.loc['K1_old', 'overall'], FINAL_EPOCHS))

In [ ]:
# Cell 8 - final model (best combo, all 70 people, 3 seeds averaged) -> submission ZIP
# CodaBench direction CONFIRMED on 13 Oct: distance-like, LOWER = same person
LOWER_IS_SAME = True
fn, vn = COMBOS[BEST]
XF = torch.tensor(build(fn, 'train'), device=dev); XV = torch.tensor(build(vn, 'train'), device=dev)
finals = [train_model(XF, XV, np.arange(len(y)), None, seed=s, max_epochs=FINAL_EPOCHS)[0] for s in SEEDS]

RUN = '05_' + BEST
OUT = '/content/sub_' + RUN
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang, fname in LANG_FILE.items():
        split = track + '_' + lang
        pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
        xf = torch.tensor(build(fn, split), device=dev); xv = torch.tensor(build(vn, split), device=dev)
        assert len(pair_ids) == len(xf) == len(xv)
        s = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
        out = -s if LOWER_IS_SAME else s
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, out):
                f.write('%s %.6f\n' % (pid, v))
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_PATH = os.path.join(SUBDIR, RUN + '.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
note('Submission ZIP saved: ' + ZIP_PATH[len(BASE):])

In [ ]:
# Cell 9 - experiment log + summary
LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:   # record the CodaBench dev scores we already know
    log['codabench_dev'] = log['codabench_dev'].astype(object)
    log.loc[log.run == '02_fop_baseline', 'codabench_dev'] = 36.79
    log.loc[log.run == '03_D_balanced', 'codabench_dev'] = 33.49
    log = log[~log.run.isin(['05_' + c for c in tab.index])]   # replace old rows of these combos (no duplicates)
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '05_' + c, 'change': 'features: face=%s voice=%s' % ('+'.join(COMBOS[c][0]), '+'.join(COMBOS[c][1])),
                     'seed': '1,2,3', 'cv_std': tab.loc[c, 'std'], 'cv_gender': tab.loc[c, 'gender'],
                     'cv_overall': tab.loc[c, 'overall'], 'codabench_dev': ''} for c in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)

print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)